In [1]:
# Will allow us to use gradient boosting for modeling
%pip install xgboost

  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 31.9 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.5/336.5 MB 80.2 MB/s  0:00:04m0:00:0100:01
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [xgboost]m2/3 [xgboost]ccl-cu12]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Will allow us to use NumPy for numerical computations
%pip install numpy


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Will allow us to to use scikit-learn for data analysis and modeling
%pip install -U scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
#Importing necessary libraries
import pandas as pd
import boto3   # AWS SDK, used to read from S3
import io
import gzip
import json
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
import numpy as np
# Metrics used to score how close the predictions are to the real speeds
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [6]:
# Shared S3 client used by the extraction functions below
s3_client = boto3.client("s3")

In [9]:
def extract_clean_tomtom_traffic_data(
    bucket: str = "s3-3601-final-project",
    prefix: str = "tomtom/traffic/",
    s3_client=None,
) -> pd.DataFrame:
    """Extracts traffic files from S3 across all paginated pages and combines them into one DataFrame."""
    if s3_client is None:
        s3_client = boto3.client("s3")

    paginator = s3_client.get_paginator("list_objects_v2")
    df_list = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith((".csv")):
                continue

            file_obj = s3_client.get_object(Bucket=bucket, Key=key)
            raw_bytes = file_obj["Body"].read()

            try:
                data_bytes = gzip.decompress(raw_bytes)
            except gzip.BadGzipFile:
                data_bytes = raw_bytes

            if key.endswith(".csv"):
                obj_df = pd.read_csv(io.BytesIO(data_bytes))
            else:
                # Handle .json.gz if any exist in the prefix
                obj_df = pd.read_json(io.BytesIO(data_bytes))

            if not obj_df.empty:
                df_list.append(obj_df)

    if not df_list:
        return pd.DataFrame()

    return pd.concat(df_list, ignore_index=True)

In [ ]:
def extract_tomtom_traffic_data():
    """
    Extracts traffic data from the specified S3 bucket and prefix, decompresses the files if necessary, and combines them into a single DataFrame.

    Args: None

    Returns:
        tomtom_df: pd.DataFrame: A DataFrame containing the combined traffic data from all relevant files in the S3 bucket.
    """
    # S3 location of the TomTom speed files saved by the speeds extraction notebook
    bucket = "s3-3601-final-project"
    prefix = "tomtom/traffic/"
    # S3 has no real folders; the prefix matches every object whose path starts with "tomtom/traffic/"
    # (a single call returns at most 1,000 objects, so use a paginator if the file count grows)
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    # Each file becomes its own DataFrame, collected here and combined later
    df_list = []
    # Download each data file and load it into its own DataFrame
    if "Contents" in response:
        for obj in response["Contents"]:
            key = obj["Key"]

            # Skip anything that isn't a data file
            if not key.endswith((".json.gz", ".csv")):
                continue

            file_obj = s3_client.get_object(Bucket=bucket, Key=key)
            raw_bytes = file_obj["Body"].read()
            # Files may or may not be gzipped, so try to decompress
            # and fall back to the raw bytes if they aren't
            try:
                data_bytes = gzip.decompress(raw_bytes)
            except gzip.BadGzipFile:
                data_bytes = raw_bytes
                
            # Parse the CSV bytes into a DataFrame
            obj_df = pd.read_csv(io.BytesIO(data_bytes))
            df_list.append(obj_df)
            # Runs on every pass of the loop (the same concat after the loop makes this one redundant)
            final_df = pd.concat(df_list, ignore_index=True)
        # After the loop, combine all files into one DataFrame
        if df_list:
            final_df = pd.concat(df_list, ignore_index=True)
    return final_df

,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,hour,month,speed_ratio,holiday,attributes.LocationID,attributes.COUNTY,attributes.ROUTE,attributes.LOCATION,attributes.AADT_2022
0,2026-09-26 19:03:46,35.307778,-80.724583,19.0,27.0,NaN,1.0,5.0,19.0,9.0,0.703704,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
1,2026-09-26 19:03:46,35.313611,-80.701944,40.0,40.0,NaN,1.0,5.0,19.0,9.0,1.000000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
2,2026-09-26 19:03:46,35.311361,-80.711222,26.0,31.0,NaN,1.0,5.0,19.0,9.0,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
3,2026-09-26 19:03:47,35.309306,-80.718056,26.0,31.0,NaN,1.0,5.0,19.0,9.0,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
4,2026-09-26 19:03:47,35.306917,-80.722639,26.0,31.0,NaN,1.0,5.0,19.0,9.0,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
5,2026-09-26 19:03:47,35.293111,-80.749361,17.0,28.0,NaN,1.0,5.0,19.0,9.0,0.607143,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
6,2026-09-26 19:03:47,35.294889,-80.742111,55.0,55.0,NaN,1.0,5.0,19.0,9.0,1.000000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
7,2026-09-26 19:03:47,35.299639,-80.745750,19.0,28.0,NaN,1.0,5.0,19.0,9.0,0.678571,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
8,2026-09-26 19:03:47,35.287972,-80.759472,20.0,25.0,NaN,1.0,5.0,19.0,9.0,0.800000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN
9,2026-09-26 19:03:47,35.301861,-80.752333,18.0,26.0,NaN,1.0,5.0,19.0,9.0,0.692308,Not a federal holiday,NaN,NaN,NaN,NaN,NaN


In [8]:
# Add time features from the timestamp so the model can learn rush hour,
# weekday/weekend, and seasonal patterns
# errors="coerce" turns unparseable values into NaT instead of raising an error
#final_df["timestamp"] = pd.to_datetime(final_df["timestamp"], errors="coerce")

# Int64 (nullable integer) keeps missing values as <NA> instead of converting to float
#final_df["hour"] = final_df["timestamp"].dt.hour.astype("Int64")
#final_df["day_of_week"] = final_df["timestamp"].dt.dayofweek.astype("Int64")
#final_df["month"] = final_df["timestamp"].dt.month.astype("Int64")
# CAN THESE BE REMOVED?!? Check speed extraction demo file for the same code. If it is there, remove it from here.

def school_holiday_check(traffic_df):
    """
    Checks if a given date falls within the defined school holiday periods (defined in the function for UNCC).

    Args:
        traffic_df (pd.DataFrame): A DataFrame containing traffic data with a 'timestamp' column to be compared to school holiday periods.

    Returns:
        traffic_df (pd.DataFrame): The input DataFrame with an additional 'school_holiday' column indicating whether each date falls within a school holiday period (1 for holiday, 0 for non-holiday).
    """
    # Make sure timestamps are datetimes before comparing them to the holiday dates
    traffic_df["timestamp"] = pd.to_datetime(traffic_df["timestamp"])
    
    # UNCC school holiday ranges as (start, end), inclusive; update these each semester
    school_holidays = [
        ("2026-10-12", "2026-10-13"), #Student recess
        ("2026-11-11", "2026-11-11"), #Veterans Day
        ("2026-11-25", "2026-11-28"),  # Thanksgiving
        ("2026-12-24", "2026-12-31")  # Winter break
    ]

    school_holidays = [
        (pd.Timestamp(start), pd.Timestamp(end))
        for start, end in school_holidays
    ]

    # Flag = 1 if the row's date falls in any holiday range, otherwise 0 (missing dates get 0)
    traffic_df["school_holiday"] = traffic_df["timestamp"].apply(
        lambda date: int(pd.notna(date) and any(start <= date.normalize() <= end
                        for start, end in school_holidays
                    )))
    
    return traffic_df

In [ ]:
def extract_ncdot_traffic_data_s3():
    """
    Extracts traffic data from the specified S3 bucket and key for NCDOT traffic data, decompresses the file if necessary, and returns it as a DataFrame with coordinates edited to match the other data (account for slight decimal differences).

    Args: None

    Returns:
        ncdot_df: pd.DataFrame: A DataFrame containing the NCDOT traffic data.
    """
    # S3 location of the NCDOT AADT file saved by the counts extraction notebook
    bucket = "s3-3601-final-project"
    # AADT = average vehicles per day, which tells the model how busy each road normally is
    s3_client = boto3.client("s3")

    # Download one specific snapshot of the NCDOT stations file (the key is its path in the bucket
    # and is hardcoded, so update it if the file is re-extracted)
    response2 = s3_client.get_object(Bucket=bucket, Key="ncdot/traffic/aadt_stations_2026-09-27_15-13-28-302554.csv")

    # Read the raw file contents
    raw_bytes = response2["Body"].read()
    # Files may or may not be gzipped, so try to decompress
    # and fall back to the raw bytes if they aren't
    try:
        data_bytes = gzip.decompress(raw_bytes)
    except gzip.BadGzipFile:
        data_bytes = raw_bytes
    # Parse the CSV bytes into a DataFrame
    ncdot_df = pd.read_csv(io.BytesIO(data_bytes))

    # NCDOT and TomTom coordinates differ slightly in their decimals, so we store the TomTom values
    # as matched_latitude/longitude, which lets each station line up with a sample point in the join later.
    # These lists are hardcoded by row position, so the file's rows must stay in the order noted below.
    matched_latitudes = [
        35.287972,  # Row 0: EAST OF STETSON DR
        35.311361,  # Row 1: WEST OF I-485
        35.293111,  # Row 2: SOUTH OF UNIVERSITY POINTE BLVD
        35.309306,  # Row 3: EAST OF SUTHER RD
        35.306917,  # Row 4: WEST OF SUTHER RD
        35.313611,  # Row 5: EAST OF SR 5394 MALLARD CK
        35.311861,  # Row 6: WEST OF W 7TH ST 
        35.301861,  # Row 7: SOUTH OF US 29
        35.294889,  # Row 8: EAST OF SR 5316
        35.299639,  # Row 9: EAST OF SR 2939
        35.307778   # Row 10: NORTH OF NC 49
    ]

    matched_longitudes = [
        -80.759472, # Row 0
        -80.711222, # Row 1
        -80.749361, # Row 2
        -80.718056, # Row 3
        -80.722639, # Row 4
        -80.701944, # Row 5
        -80.744000, # Row 6 
        -80.752333, # Row 7
        -80.742111, # Row 8
        -80.745750, # Row 9
        -80.724583  # Row 10
    ]

    # Add the matched coordinates as new columns, row by row (created on first assignment)
    for i in range(len(matched_latitudes)):
        ncdot_df.at[i, "matched_latitude"] = matched_latitudes[i]
        ncdot_df.at[i, "matched_longitude"] = matched_longitudes[i]

    return ncdot_df

,LocationID,County,Route,Location,AADT_2022,longitude,latitude
0,600000044,MECKLENBURG,US 29/NC 49,EAST OF STETSON DR,29500,-80.759574,35.287985
1,600000063,MECKLENBURG,NC 49,WEST OF I-485,32500,-80.711385,35.311292
2,600000065,MECKLENBURG,US 29,SOUTH OF UNIVERSITY POINTE BLVD,18000,-80.755360,35.294440
3,600000107,MECKLENBURG,NC 49,EAST OF SUTHER RD,26500,-80.727801,35.303270
4,600000108,MECKLENBURG,NC 49,WEST OF SUTHER RD,27500,-80.729977,35.301876
5,600000145,MECKLENBURG,NC 49,EAST OF SR 5394 MALLARD CK CH RD,42000,-80.717633,35.309346
6,600000305,MECKLENBURG,US 29,WEST OF W 7TH ST,17000,-80.844289,35.232562
7,600000412,MECKLENBURG,NC 24 (WT HARRIS BLVD),SOUTH OF US 29,47500,-80.748333,35.304176
8,600000414,MECKLENBURG,NC 24 (E WT HARRIS BLVD),EAST OF SR 5316,51500,-80.739913,35.293191
9,600000829,MECKLENBURG,NC 49,EAST OF SR 2939,32500,-80.722975,35.306539


In [ ]:
def extract_ncdot_traffic_data(ncdot_df):
    """
    For a provided dataframe of NCDOT traffic data, edits the coordinates to match the TomTom data perfectly (account for slight decimal differences).
    Pared down to only the coordinate matching part of the S3 extraction function, so it can be used on any dataframe of NCDOT traffic data.

    Args: 
        ncdot_df (pd.DataFrame): A DataFrame containing the NCDOT traffic data.
    Returns:
        ncdot_df (pd.DataFrame): The input DataFrame with updated coordinates to match the TomTom data.
    """
    # Work on a copy so the caller's DataFrame isn't modified
    ncdot_df = ncdot_df.copy()

    # Hardcoded by row position, so the rows must stay in the order noted below
    matched_latitudes = [
        35.287972,  # Row 0: EAST OF STETSON DR
        35.311361,  # Row 1: WEST OF I-485
        35.293111,  # Row 2: SOUTH OF UNIVERSITY POINTE BLVD
        35.309306,  # Row 3: EAST OF SUTHER RD
        35.306917,  # Row 4: WEST OF SUTHER RD
        35.313611,  # Row 5: EAST OF SR 5394 MALLARD CK
        35.311861,  # Row 6: WEST OF W 7TH ST 
        35.301861,  # Row 7: SOUTH OF US 29
        35.294889,  # Row 8: EAST OF SR 5316
        35.299639,  # Row 9: EAST OF SR 2939
        35.307778   # Row 10: NORTH OF NC 49
    ]

    matched_longitudes = [
        -80.759472, # Row 0
        -80.711222, # Row 1
        -80.749361, # Row 2
        -80.718056, # Row 3
        -80.722639, # Row 4
        -80.701944, # Row 5
        -80.744000, # Row 6 
        -80.752333, # Row 7
        -80.742111, # Row 8
        -80.745750, # Row 9
        -80.724583  # Row 10
    ]

    # Add the matched coordinates as new columns, row by row (created on first assignment)
    for i in range(len(matched_latitudes)):
        ncdot_df.at[i, "matched_latitude"] = matched_latitudes[i]
        ncdot_df.at[i, "matched_longitude"] = matched_longitudes[i]

    return ncdot_df

In [ ]:
def combine_dataframes(tomtom_df, ncdot_df):
    """
    Combines two DataFrames (tomtom_df and ncdot_df) by matching latitude and longitude values, and adds the AADT_2022 column from ncdot_df to tomtom_df.

    Args:
        tomtom_df (pd.DataFrame): The first DataFrame containing traffic data with latitude and longitude columns.
        ncdot_df (pd.DataFrame): The second DataFrame containing traffic data with matched_latitude, matched_longitude, and AADT_2022 columns.

    Returns:
        traffic_df (pd.DataFrame): A combined DataFrame with the AADT_2022 column added from ncdot_df to tomtom_df.
    """

    # Make sure all coordinate columns are numeric so rounding works
    tomtom_df["latitude"] = pd.to_numeric(tomtom_df["latitude"])
    tomtom_df["longitude"] = pd.to_numeric(tomtom_df["longitude"])
    ncdot_df["matched_latitude"] = pd.to_numeric(ncdot_df["matched_latitude"])
    ncdot_df["matched_longitude"] = pd.to_numeric(ncdot_df["matched_longitude"])
    # Build a join key from coordinates rounded to 4 decimals (~11 m) so tiny float differences still match
    ncdot_df["round_key"] = (
        ncdot_df["matched_latitude"].round(4).astype(str) + "_" + 
        ncdot_df["matched_longitude"].round(4).astype(str)
    )
    # Lookup table: round_key -> AADT_2022 (assumes each key appears once)
    traffic_map=ncdot_df.set_index("round_key")["AADT_2022"]

    tomtom_df["round_key"] = (
        tomtom_df["latitude"].round(4).astype(str) + "_" + 
        tomtom_df["longitude"].round(4).astype(str)
    )
    # Attach each station's AADT to every matching TomTom row (unmatched rows get NaN)
    tomtom_df["AADT_2022"] = tomtom_df["round_key"].map(traffic_map)

    # Remove the temporary join keys
    tomtom_df = tomtom_df.drop(columns=["round_key"])
    ncdot_df = ncdot_df.drop(columns=["round_key"])
    combined_traffic_df = tomtom_df.copy()  # Create a copy of tomtom_df to avoid modifying the original DataFrame

    return combined_traffic_df

,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,hour,month,speed_ratio,holiday,attributes.LocationID,attributes.COUNTY,attributes.ROUTE,attributes.LOCATION,attributes.AADT_2022,school_holiday,AADT_2022
0,2026-09-26 19:03:46,35.307778,-80.724583,19.0,27.0,NaN,1.0,5,19,9,0.703704,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,15500.0
1,2026-09-26 19:03:46,35.313611,-80.701944,40.0,40.0,NaN,1.0,5,19,9,1.000000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,42000.0
2,2026-09-26 19:03:46,35.311361,-80.711222,26.0,31.0,NaN,1.0,5,19,9,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,32500.0
3,2026-09-26 19:03:47,35.309306,-80.718056,26.0,31.0,NaN,1.0,5,19,9,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,26500.0
4,2026-09-26 19:03:47,35.306917,-80.722639,26.0,31.0,NaN,1.0,5,19,9,0.838710,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,27500.0
5,2026-09-26 19:03:47,35.293111,-80.749361,17.0,28.0,NaN,1.0,5,19,9,0.607143,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,18000.0
6,2026-09-26 19:03:47,35.294889,-80.742111,55.0,55.0,NaN,1.0,5,19,9,1.000000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,51500.0
7,2026-09-26 19:03:47,35.299639,-80.745750,19.0,28.0,NaN,1.0,5,19,9,0.678571,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,32500.0
8,2026-09-26 19:03:47,35.287972,-80.759472,20.0,25.0,NaN,1.0,5,19,9,0.800000,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,29500.0
9,2026-09-26 19:03:47,35.301861,-80.752333,18.0,26.0,NaN,1.0,5,19,9,0.692308,Not a federal holiday,NaN,NaN,NaN,NaN,NaN,0,47500.0


In [11]:
def preprocess_train_data(combined_df):
    """
    Preprocesses the combined DataFrame by cleaning the data, creating new features, and preparing it for machine learning. Trains an XGBoost regression model on the preprocessed data.

    Args:
        combined_df (pd.DataFrame): The combined DataFrame containing all the data.

    Returns:
        model (xgb.XGBRegressor): The trained XGBoost regression model.
        X_test (pd.DataFrame): Held-out test features (the last 20% of rows by time).
        y_test (pd.Series): Actual speeds for the test rows.
    """
    final_df = combined_df.copy()
    # Give each distinct (latitude, longitude) point an integer ID the model can use as a feature
    final_df["location_id"] = final_df.groupby(["latitude", "longitude"]).ngroup()

    # Sort oldest to newest so the train/test split below is chronological
    final_df = final_df.sort_values("timestamp").reset_index(drop=True)

    # 1 if the date is a US federal holiday, 0 otherwise
    final_df["is_holiday"] = (final_df["holiday"] != "Not a federal holiday").astype(int)

    # Model inputs (features) and the value we want to predict (target)
    feature_cols = ["location_id", "day_of_week", "hour", "month", "school_holiday", "free_flow_speed", "is_holiday", "AADT_2022"]
    target_col = "current_speed"

    # Drop rows with any missing feature or target value
    valid_rows = final_df[feature_cols].notna().all(axis=1) & final_df[target_col].notna()
    df_clean = final_df.loc[valid_rows].copy()

    # Separate the features (X) from the target (y)
    X = df_clean[feature_cols]
    y = df_clean[target_col]

    # Chronological 80/20 split: train on earlier rows, test on later ones (no shuffling, so no future data leaks into training)
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    # Gradient-boosted trees: 100 trees, max depth 5, learning rate 0.1
    model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1)
    model.fit(X_train, y_train)

    return model, X_test, y_test

In [12]:
def evaluate_model(model, X_test, y_test):
    """
    Evaluates the performance of the trained model on the test set and prints various performance metrics.
    
    Args:
        model: xgb.XGBRegressor: The trained XGBoost regression model.
        X_test (pd.DataFrame): The test set features.
        y_test (pd.Series): The actual target values for the test set.
    Returns:
        None, it displays the evaluation metrics and predicted outputs for some inputs in the test set.
    """
    # Predict speeds for the held-out test rows
    y_pred = model.predict(X_test)

    # Shows the average absolute error between predicted and actual speeds
    mae = mean_absolute_error(y_test, y_pred) 
    # Shows the root mean squared error between predicted and actual speeds
    rmse = np.sqrt(mean_squared_error(y_test, y_pred)) 
    # Shows the proportion of variance in the target variable explained by the model
    r2 = r2_score(y_test, y_pred) 

    # Swap zero actual speeds for a tiny value so MAPE never divides by zero
    y_test_stable = np.where(y_test == 0, 1e-5, y_test)
    # Shows the mean absolute percentage error between predicted and actual speeds
    mape = np.mean(np.abs((y_test - y_pred) / y_test_stable)) * 100 

    print("Model Performance:")
    print(f"Mean Absolute Error (MAE): {mae:.2f} mph")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f} mph")
    print(f"MAPE (Percentage Error): {mape:.2f}%")
    print(f"R² Score (Variance Explained): {r2:.4f}")

    # Show the inputs and predicted speed for the first four test rows
    for i in range(4):
        print(f"Input:\n{X_test.iloc[i]} \nOutput speed: {y_pred[i]} mph")

In [14]:
combined_df = extract_clean_tomtom_traffic_data()
model, X_test, y_test = preprocess_train_data(combined_df)

print(evaluate_model(model, X_test, y_test))

Model Performance:
Mean Absolute Error (MAE): 1.85 mph
Root Mean Squared Error (RMSE): 2.76 mph
MAPE (Percentage Error): 7.74%
R² Score (Variance Explained): 0.9167
Input:
location_id           13.0
day_of_week            2.0
hour                  14.0
month                 11.0
school_holiday         1.0
free_flow_speed       32.0
is_holiday             1.0
AADT_2022          27500.0
Name: 30716, dtype: float64 
Output speed: 29.32106590270996 mph
Input:
location_id            2.0
day_of_week            2.0
hour                  14.0
month                 11.0
school_holiday         1.0
free_flow_speed       28.0
is_holiday             1.0
AADT_2022          18000.0
Name: 30717, dtype: float64 
Output speed: 26.644533157348633 mph
Input:
location_id            7.0
day_of_week            2.0
hour                  14.0
month                 11.0
school_holiday         1.0
free_flow_speed       25.0
is_holiday             1.0
AADT_2022          47500.0
Name: 30718, dtype: float64 
Output

In [15]:
model_filename = "model.json"
model.save_model(model_filename)

s3= boto3.client("s3")
bucket_name = "s3-3601-final-project"
s3_key = "models/v1/model.json"

s3.upload_file(model_filename, bucket_name, s3_key)
print(f"Uploaded to s3://{bucket_name}/{s3_key}")

Uploaded to s3://s3-3601-final-project/models/v1/model.json


In [17]:
import xgboost as xgb

s3.download_file(bucket_name, s3_key, "downloaded_model.json")

reloaded_model = xgb.XGBRegressor()  # or XGBClassifier()
reloaded_model.load_model("downloaded_model.json")
if reloaded_model is not None:
    print("Success!")

Success!


In [21]:
predict_custom_traffic(reloaded_model, 3, "11-01-26 22:00:00")


--- Traffic Prediction Results ---
Location ID: 3
Timestamp: 11-01-26 22:00:00
Free-Flow Speed: 31.0 mph
AADT (Volume): 26500.0
Predicted Speed: 23.62 mph
Speed Index: 0.76
Traffic Condition: Medium Traffic


(np.float32(23.617205), 'Medium Traffic')

In [27]:
from traffic_predictor import predict_custom_traffic

predict_custom_traffic(reloaded_model, 5, "11-01-26 22:00:00")


--- Traffic Prediction Results ---
Location ID: 5
Timestamp: 11-01-26 22:00:00
Free-Flow Speed: 28.0 mph
AADT (Volume): 18000.0
Predicted Speed: 24.97 mph
Speed Index: 0.89
Traffic Condition: Low Traffic


(np.float32(24.974167), 'Low Traffic')

In [13]:
def predict_custom_traffic(model, location_id, timestamp_str):
    """
    Takes a single location_id and timestamp string, automatically assigns the hardcoded free_flow_speed and AADT_2022, runs feature engineering, and predicts traffic speed and congestion level.
    
    Args:
        model: xgb.XGBRegressor: The trained XGBoost regression model.
        location_id (int): The location ID for which to predict traffic speed.
        timestamp_str (str): The timestamp string in the format 'YYYY-MM-DD HH:MM:SS' for which to predict traffic speed.
    Returns:
        tuple: A tuple containing the predicted traffic speed (float) and congestion level (str).
    """

    # Constants per location (free-flow speed in mph and AADT) so the user only supplies a place and time.
    # location_id comes from groupby().ngroup() in training, so these IDs only match if the training data is the same.
    LOCATION_PARAMS = {
    0: {"free_flow_speed": 27.0, "AADT_2022": 15500.0},
    1: {"free_flow_speed": 40.0, "AADT_2022": 42000.0},
    2: {"free_flow_speed": 31.0, "AADT_2022": 32500.0},
    3: {"free_flow_speed": 31.0, "AADT_2022": 26500.0},
    4: {"free_flow_speed": 31.0, "AADT_2022": 27500.0},
    5: {"free_flow_speed": 28.0, "AADT_2022": 18000.0},
    6: {"free_flow_speed": 55.0, "AADT_2022": 51500.0},
    7: {"free_flow_speed": 28.0, "AADT_2022": 32500.0},
    8: {"free_flow_speed": 25.0, "AADT_2022": 29500.0},
    9: {"free_flow_speed": 26.0, "AADT_2022": 47500.0},
    10: {"free_flow_speed": 29.0, "AADT_2022": 17000.0}
    }
    
    # 1. Validate or retrieve hardcoded parameters for the location
    if location_id not in LOCATION_PARAMS:
        raise ValueError(f"Unknown location_id: {location_id}. Choose between 0 and 10.")
    
    params = LOCATION_PARAMS[location_id]
    
    # 2. Build a single-row DataFrame for inference
    df = pd.DataFrame([{
        "location_id": location_id,
        "timestamp": timestamp_str,
        "free_flow_speed": params["free_flow_speed"],
        "AADT_2022": params["AADT_2022"]
    }])
    
    # 3. Parse timestamp and extract temporal features
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["month"] = df["timestamp"].dt.month
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["hour"] = df["timestamp"].dt.hour
    
    # 4. Flag school holidays (UNCC calendar)
    df = school_holiday_check(df)
    df["is_holiday"] = df["school_holiday"] # Approximation: training used federal holidays, but the model expects this column
    
    # 5. Select the features (same columns and order as training) and predict
    feature_cols = ["location_id", "day_of_week", "hour", "month", "school_holiday", "free_flow_speed", "is_holiday", "AADT_2022"]
    X_inference = df[feature_cols]
    
    predicted_speed = model.predict(X_inference)[0]
    
    # 6. Speed index = predicted / free-flow speed: >= 0.8 Low, >= 0.5 Medium, otherwise High traffic
    free_flow = params["free_flow_speed"]
    speed_index = predicted_speed / free_flow
    
    if speed_index >= 0.8:
        traffic_level = "Low Traffic"
    elif speed_index >= 0.5:
        traffic_level = "Medium Traffic"
    else:
        traffic_level = "High Traffic"
        
    print(f"\n--- Traffic Prediction Results ---")
    print(f"Location ID: {location_id}")
    print(f"Timestamp: {timestamp_str}")
    print(f"Free-Flow Speed: {free_flow} mph")
    print(f"AADT (Volume): {params['AADT_2022']}")
    print(f"Predicted Speed: {predicted_speed:.2f} mph")
    print(f"Speed Index: {speed_index:.2f}")
    print(f"Traffic Condition: {traffic_level}")
    
    return predicted_speed, traffic_level

In [25]:
predict_custom_traffic(model, 4, "2026-11-23 20:10:0")


--- Traffic Prediction Results ---
Location ID: 4
Timestamp: 2026-11-23 20:10:0
Free-Flow Speed: 31.0 mph
AADT (Volume): 27500.0
Predicted Speed: 24.21 mph
Speed Index: 0.78
Traffic Condition: Medium Traffic


(np.float32(24.20842), 'Medium Traffic')

In [ ]:
# Run all of the functions in order to extract, preprocess, and evaluate the traffic data

# 1. Pull the TomTom speed files from S3 and flag school holidays
extracted_tomtom_df = extract_tomtom_traffic_data()
extracted_tomtom_df = school_holiday_check(extracted_tomtom_df)
# 2. Load the NCDOT traffic volumes and merge them onto the TomTom rows by location
extracted_ncdot_df = extract_ncdot_traffic_data_s3()
combined_traffic_df = combine_dataframes(extracted_tomtom_df, extracted_ncdot_df)
# 3. Train the XGBoost model, then score it on the held-out test rows
preprocessed_model, X_test, y_test = preprocess_train_data(combined_traffic_df)
evaluate_model(preprocessed_model, X_test, y_test)